In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset

REPO_ID = "witgaw/METR-LA"

train = load_dataset(
    REPO_ID,
    split="train"
)

assert train.num_rows > 0
assert "node_id" in train.column_names
assert "t0_timestamp" in train.column_names

print(train)
print("\nDataset loaded successfully.")

In [ ]:
columns = train.column_names

if "x_t+0_d0" in columns:
    SPEED_COL = "x_t+0_d0"

elif "x_t-0_d0" in columns:
    SPEED_COL = "x_t-0_d0"

else:
    raise AssertionError(
        "Current-time speed column not found."
    )

print("Current speed column:", SPEED_COL)

assert SPEED_COL in columns

In [ ]:
temporal_df = (
    train
    .select_columns([
        "node_id",
        "t0_timestamp",
        SPEED_COL
    ])
    .to_pandas()
)

temporal_df = temporal_df.rename(
    columns={SPEED_COL: "speed"}
)

temporal_df["t0_timestamp"] = pd.to_datetime(
    temporal_df["t0_timestamp"]
)

print("Rows:", len(temporal_df))
print("Sensors:", temporal_df["node_id"].nunique())

print(
    "Collection window:",
    temporal_df["t0_timestamp"].min(),
    "to",
    temporal_df["t0_timestamp"].max()
)

assert temporal_df["node_id"].nunique() == 207
assert temporal_df["t0_timestamp"].notna().all()

print("\nPASS: Diagnostic dataframe created.")
print("Original train dataset remains unmodified.")

In [ ]:
timestamps = pd.Series(
    temporal_df["t0_timestamp"].unique()
).sort_values().reset_index(drop=True)

print("Unique timestamps:", len(timestamps))

print("First timestamp:", timestamps.iloc[0])
print("Last timestamp :", timestamps.iloc[-1])

assert len(timestamps) > 1
assert timestamps.is_monotonic_increasing
assert timestamps.is_unique

In [ ]:
expected_timestamps = pd.date_range(
    start=timestamps.iloc[0],
    end=timestamps.iloc[-1],
    freq="5min"
)

missing_timestamps = expected_timestamps.difference(
    pd.DatetimeIndex(timestamps)
)

print("=" * 65)
print("5-MINUTE TIMESTAMP COVERAGE")
print("=" * 65)

print("Expected timestamps :", len(expected_timestamps))
print("Actual timestamps   :", len(timestamps))
print("Missing intervals   :", len(missing_timestamps))

if len(missing_timestamps) == 0:

    print(
        "\nPASS: No missing 5-minute timestamps "
        "within the observed collection window."
    )

else:

    print(
        f"\nWARNING: {len(missing_timestamps)} expected "
        "5-minute timestamp(s) are absent."
    )

    print("\nFirst 20 missing timestamps:")

    for ts in missing_timestamps[:20]:
        print(" ", ts)

In [ ]:
timestamp_diffs = timestamps.diff()

largest_gap = timestamp_diffs.max()

largest_gap_idx = timestamp_diffs.idxmax()

gap_end = timestamps.iloc[largest_gap_idx]
gap_start = timestamps.iloc[largest_gap_idx - 1]

print("=" * 65)
print("LARGEST TIMESTAMP GAP")
print("=" * 65)

print("Gap start :", gap_start)
print("Gap end   :", gap_end)
print("Gap size  :", largest_gap)

if largest_gap == pd.Timedelta(minutes=5):

    print(
        "\nPASS: Largest observed gap is exactly 5 minutes."
    )

else:

    missing_steps_inside_gap = (
        int(
            largest_gap
            / pd.Timedelta(minutes=5)
        ) - 1
    )

    print(
        "\nWARNING: A gap larger than the expected "
        "5-minute sampling interval was detected."
    )

    print(
        "Missing 5-minute positions inside largest gap:",
        missing_steps_inside_gap
    )

In [ ]:
off_grid = timestamps[
    (
        (timestamps.dt.minute % 5 != 0)
        | (timestamps.dt.second != 0)
    )
]

print("Off-grid timestamps:", len(off_grid))

if len(off_grid) == 0:

    print("PASS: All timestamps lie on 5-minute boundaries.")

else:

    print(
        "WARNING: Some timestamps are not aligned "
        "to 5-minute boundaries."
    )

    print(off_grid.head(20))

In [ ]:
calendar = pd.DataFrame({
    "timestamp": timestamps
})

calendar["day_of_week"] = (
    calendar["timestamp"].dt.dayofweek
)

calendar["day_name"] = (
    calendar["timestamp"].dt.day_name()
)

calendar["is_weekend"] = (
    calendar["day_of_week"] >= 5
)

calendar["day_type"] = np.where(
    calendar["is_weekend"],
    "Weekend",
    "Weekday"
)

display(calendar.head())

assert set(calendar["day_type"].unique()).issubset(
    {"Weekday", "Weekend"}
)

In [ ]:
day_type_counts = (
    calendar["day_type"]
    .value_counts()
)

day_type_percent = (
    calendar["day_type"]
    .value_counts(normalize=True)
    .mul(100)
)

coverage_summary = pd.DataFrame({
    "timestamps": day_type_counts,
    "percentage": day_type_percent
})

display(coverage_summary)

weekend_count = int(
    (calendar["day_type"] == "Weekend").sum()
)

weekday_count = int(
    (calendar["day_type"] == "Weekday").sum()
)

weekend_fraction = (
    weekend_count / len(calendar)
)

print(f"Weekday timestamps : {weekday_count:,}")
print(f"Weekend timestamps : {weekend_count:,}")
print(f"Weekend fraction   : {weekend_fraction:.2%}")

assert weekend_count > 0, (
    "No weekend observations found."
)

assert weekday_count > 0, (
    "No weekday observations found."
)

In [ ]:
EXPECTED_WEEKEND_FRACTION = 2 / 7

difference = abs(
    weekend_fraction
    - EXPECTED_WEEKEND_FRACTION
)

print("=" * 65)
print("WEEKEND COVERAGE CHECK")
print("=" * 65)

print(
    f"Expected calendar proportion : "
    f"{EXPECTED_WEEKEND_FRACTION:.2%}"
)

print(
    f"Observed weekend proportion  : "
    f"{weekend_fraction:.2%}"
)

print(
    f"Absolute difference          : "
    f"{difference:.2%}"
)


if difference <= 0.03:

    print(
        "\nPASS: Weekend coverage is roughly consistent "
        "with normal calendar proportions."
    )

elif difference <= 0.08:

    print(
        "\nWARNING: Weekend representation differs "
        "noticeably from normal calendar proportions."
    )

else:

    print(
        "\nSTRONG WARNING: Weekend representation is "
        "highly abnormal."
    )

    print(
        "This could indicate weekend exclusion, "
        "subsampling, or another preprocessing change."
    )

In [ ]:
temporal_df["hour"] = (
    temporal_df["t0_timestamp"].dt.hour
)

temporal_df["is_weekend"] = (
    temporal_df["t0_timestamp"].dt.dayofweek >= 5
)

temporal_df["day_type"] = np.where(
    temporal_df["is_weekend"],
    "Weekend",
    "Weekday"
)

assert temporal_df["hour"].between(0, 23).all()

In [ ]:
hourly_speed = (
    temporal_df
    .groupby(
        ["day_type", "hour"],
        as_index=False
    )
    .agg(
        average_speed=("speed", "mean")
    )
)

hourly_pivot = hourly_speed.pivot(
    index="hour",
    columns="day_type",
    values="average_speed"
)

display(hourly_pivot)

assert "Weekday" in hourly_pivot.columns
assert "Weekend" in hourly_pivot.columns

assert len(hourly_pivot) == 24

In [ ]:
ax = hourly_pivot.plot(
    figsize=(12, 6),
    marker="o"
)

ax.set_title(
    "METR-LA Average Speed by Hour of Day"
)

ax.set_xlabel("Hour of Day")
ax.set_ylabel("Average Speed (mph)")

ax.set_xticks(range(24))

ax.grid(
    True,
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
weekday_hourly = hourly_pivot["Weekday"]

morning_rush = weekday_hourly.loc[7:9].mean()
evening_rush = weekday_hourly.loc[16:18].mean()

late_night = pd.concat([
    weekday_hourly.loc[0:5],
    weekday_hourly.loc[22:23]
]).mean()

print("=" * 65)
print("WEEKDAY RUSH-HOUR DIAGNOSTIC")
print("=" * 65)

print(f"Morning rush avg speed : {morning_rush:.2f} mph")
print(f"Evening rush avg speed : {evening_rush:.2f} mph")
print(f"Late-night avg speed   : {late_night:.2f} mph")

if (
    morning_rush < late_night
    and evening_rush < late_night
):

    print(
        "\nPASS: Weekday rush-hour speeds are lower "
        "than late-night speeds."
    )

else:

    print(
        "\nWARNING: Expected weekday rush-hour dip "
        "is not clearly present."
    )

    print(
        "Inspect the hourly plot before drawing "
        "a traffic-pattern conclusion."
    )

In [ ]:
temporal_df["day_of_week"] = (
    temporal_df["t0_timestamp"].dt.dayofweek
)

temporal_df["day_name"] = (
    temporal_df["t0_timestamp"].dt.day_name()
)

dow_speed = (
    temporal_df
    .groupby(
        ["day_of_week", "day_name"],
        as_index=False
    )
    .agg(
        average_speed=("speed", "mean")
    )
    .sort_values("day_of_week")
)

display(dow_speed)

assert len(dow_speed) == 7

In [ ]:
ax = (
    dow_speed
    .set_index("day_name")["average_speed"]
    .plot.bar(
        figsize=(11, 6)
    )
)

ax.set_title(
    "METR-LA Average Speed by Day of Week"
)

ax.set_xlabel("Day of Week")
ax.set_ylabel("Average Speed (mph)")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
temporal_df["date"] = (
    temporal_df["t0_timestamp"].dt.date
)

daily_speed = (
    temporal_df
    .groupby("date", as_index=False)
    .agg(
        average_speed=("speed", "mean"),
        observations=("speed", "size")
    )
)

daily_speed["date"] = pd.to_datetime(
    daily_speed["date"]
)

print("Calendar days:", len(daily_speed))

display(daily_speed.head())

assert len(daily_speed) > 0

In [ ]:
ax = daily_speed.plot(
    x="date",
    y="average_speed",
    figsize=(14, 6),
    legend=False
)

ax.set_title(
    "METR-LA Daily Average Speed Across Collection Window"
)

ax.set_xlabel("Date")
ax.set_ylabel("Average Speed (mph)")

ax.grid(
    True,
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
daily_mean = daily_speed[
    "average_speed"
].mean()

daily_std = daily_speed[
    "average_speed"
].std()

assert daily_std > 0

daily_speed["z_score"] = (
    daily_speed["average_speed"]
    - daily_mean
) / daily_std

anomalous_days = (
    daily_speed[
        daily_speed["z_score"].abs() >= 2.5
    ]
    .copy()
    .sort_values(
        "z_score",
        key=abs,
        ascending=False
    )
)

print("=" * 65)
print("POTENTIALLY ANOMALOUS CALENDAR DAYS")
print("=" * 65)

if anomalous_days.empty:

    print(
        "No dates exceed the ±2.5 standard-deviation "
        "diagnostic threshold."
    )

else:

    print(
        f"{len(anomalous_days)} potentially unusual "
        "date(s) detected.\n"
    )

    display(
        anomalous_days[
            [
                "date",
                "average_speed",
                "observations",
                "z_score"
            ]
        ]
    )

    print(
        "\nWARNING: These dates are diagnostic flags only."
    )

    print(
        "They may represent holidays, unusual traffic, "
        "sensor problems, or collection irregularities."
    )

    print(
        "No rows have been removed."
    )

In [ ]:
median_daily_obs = daily_speed[
    "observations"
].median()

daily_speed["coverage_ratio"] = (
    daily_speed["observations"]
    / median_daily_obs
)

low_coverage_days = daily_speed[
    daily_speed["coverage_ratio"] < 0.90
].copy()

print("=" * 65)
print("DAILY DATA-COVERAGE CHECK")
print("=" * 65)

print(
    "Median observations per day:",
    int(median_daily_obs)
)

if low_coverage_days.empty:

    print(
        "\nPASS: No day has less than 90% of "
        "the median daily observation count."
    )

else:

    print(
        f"\nWARNING: {len(low_coverage_days)} day(s) "
        "have reduced observation coverage."
    )

    display(
        low_coverage_days[
            [
                "date",
                "observations",
                "coverage_ratio",
                "average_speed"
            ]
        ]
    )

In [ ]:
tests = {
    "Timestamps are unique":
        timestamps.is_unique,

    "Timestamps are ordered":
        timestamps.is_monotonic_increasing,

    "All timestamps on 5-minute grid":
        len(off_grid) == 0,

    "Weekend data exists":
        weekend_count > 0,

    "Weekday data exists":
        weekday_count > 0,

    "All 7 weekdays represented":
        len(dow_speed) == 7,

    "All 24 hours represented":
        len(hourly_pivot) == 24,

    "Source dataset still intact":
        train.num_rows == 4_962_618
}


print("=" * 65)
print("TEMPORAL DIAGNOSTIC VALIDATION")
print("=" * 65)

for name, passed in tests.items():

    print(
        f"{'PASS' if passed else 'FAIL':4} : {name}"
    )

print("=" * 65)

print(f"""
Collection start        : {timestamps.iloc[0]}
Collection end          : {timestamps.iloc[-1]}
Unique timestamps       : {len(timestamps):,}
Missing 5-min positions : {len(missing_timestamps):,}
Largest observed gap    : {largest_gap}

Weekday timestamps      : {weekday_count:,}
Weekend timestamps      : {weekend_count:,}
Weekend proportion      : {weekend_fraction:.2%}

Potential unusual days  : {len(anomalous_days)}
Low-coverage days       : {len(low_coverage_days)}
""")

if all(tests.values()):

    print("ALL STRUCTURAL TEST CASES PASSED")

else:

    failed = [
        name
        for name, passed in tests.items()
        if not passed
    ]

    print("STRUCTURAL WARNING")

    for name in failed:
        print(" -", name)


print(
    "\nDIAGNOSTIC ONLY: No observations were "
    "dropped, filtered, imputed, or altered."
)